# Stage 3, first GRPO run

Runtime: **T4 GPU** (Runtime -> Change runtime type -> T4). Run the cells in order; cell 3 has to be green before cell 6 is worth the hours.


In [ ]:
# Cell 1 — the stack. Unsloth pulls a matched torch/transformers set for the
# T4. On its own it resolves TRL to 0.24.0, whose GRPOTrainer import dies on
# mergekit and llm_blender, so TRL is pinned on top to 1.13.0, the version
# the CPU smoke runs on.
!pip -q install "unsloth[colab-new]" "trl>=0.14" "peft>=0.13" "datasets>=3.0"
!pip -q install "trl==1.13.0" "safetensors>=0.8" "pydantic>=2.12,<3"
import torch; print(torch.__version__, torch.cuda.get_device_name(0))


In [ ]:
# Cell 2 — the rig, and Drive for checkpoints.
#
# For a private fork, optionally add a GitHub fine-grained token in Colab's
# Secrets panel (the key icon on the left) as GITHUB_TOKEN with notebook access
# on. Otherwise the public repository is cloned directly, or unpacked from a
# bundle at /content/drive/MyDrive/boolean-search-rl/rig.zip (`python -m
# cloris_boolean_rl.train.colab_first_run --bundle`).
from google.colab import drive; drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/boolean-search-rl
try:
    from google.colab import userdata; TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    TOKEN = None
CLONE_URL = "https://github.com/sam-vangelos/boolean-search-rl.git".replace("https://", f"https://x-access-token:{TOKEN}@") if TOKEN else "https://github.com/sam-vangelos/boolean-search-rl.git"
print("clone via", "GITHUB_TOKEN secret" if TOKEN else "public HTTPS (or rig.zip fallback)")
!git clone -q $CLONE_URL /content/boolean-search-rl 2>/dev/null \
  || (cd /content/boolean-search-rl 2>/dev/null && git pull -q $CLONE_URL main 2>/dev/null) \
  || (mkdir -p /content/boolean-search-rl && unzip -oq /content/drive/MyDrive/boolean-search-rl/rig.zip -d /content/boolean-search-rl)
%cd /content/boolean-search-rl
!git remote set-url origin https://github.com/sam-vangelos/boolean-search-rl.git 2>/dev/null || true
!ls cloris_boolean_rl/scorer/VENDORED_FROM.json && python -c "import json;d=json.load(open('cloris_boolean_rl/scorer/VENDORED_FROM.json'));print('scorer vendored at', d['commit'])"


In [ ]:
# Cell 3 — the reward has to pass its own tests before it pays for anything.
!pip -q install pytest
!python -m pytest tests/ -q
!python tools/build_split.py --check && python tools/build_prompts.py --check


In [ ]:
# Cell 4 — what the held-out prompts are worth before any training.
!python tools/run_eval.py --out runs/eval/before.json


In [ ]:
# Cell 5 — two steps end to end, then the question that decides the run.
!python -m cloris_boolean_rl.train.grpo --smoke

# GRPO learns from the SPREAD inside the group of G completions. If the base
# model cannot produce a legal single OR group at all, every sample floors,
# reward_std is zero, and 100 steps buy nothing — which is exactly what the
# 0.5B smoke shows on CPU (reward -13.82, reward_std 0, frac_reward_zero_std 1).
# So measure the 1.5B's contract pass rate on eight held-out prompts before
# spending the hours.
!python tools/run_eval.py --model Qwen/Qwen2.5-1.5B-Instruct \
    --limit 8 --samples 8 --temperature 0.9 --out runs/eval/base_probe.json
import json
probe = json.load(open("runs/eval/base_probe.json"))["systems"]["base"]
print(json.dumps(probe["gates"], indent=2), probe["diversity"])
ok = probe["gates"].get("ok", 0)
print(f"\ncontract pass rate: {ok}/8 prompts have a legal best sample")
print("STOP and add a format warm-up if this is 0" if ok == 0 else "there is a gradient; continue")


In [ ]:
# Cell 6 — the run. ~300 prompts, G=8, one epoch, checkpoints to Drive.
# Budget: a T4 generates roughly 16x160 tokens a step; 150 steps is several
# hours. Start with --max-steps 100 and resume from a checkpoint if Colab
# drops the session. Unsloth runs with fp16 off; grpo.py says why.
# A dropped session resumes with --resume-from runs/re-v1/checkpoint-<N>.
!python -m cloris_boolean_rl.train.grpo \
    --model Qwen/Qwen2.5-1.5B-Instruct \
    --prompts 300 --generations 8 --epochs 1 --max-steps 100 \
    --temperature 0.9 --beta 0.04 --save-steps 25 \
    --output-dir runs/re-v1
!cp -r runs/re-v1 /content/drive/MyDrive/boolean-search-rl/
# Run 2 (2026-09-22) used --learning-rate 5e-5 --warmup-steps 10 --lr-scheduler cosine --grad-accum 4 --max-steps 200 --max-completion-length 128; see the plan addendum.


In [ ]:
# Cell 7 — base against checkpoint, eight samples each, held-out prompts only.
!python tools/run_eval.py \
    --model Qwen/Qwen2.5-1.5B-Instruct \
    --adapter runs/re-v1/adapter \
    --samples 8 --temperature 0.8 \
    --out runs/eval/after.json \
    --note "first GRPO run, re-v1 split"
import json; print(json.dumps(json.load(open('runs/eval/after.json'))['win_rates'], indent=2))


In [ ]:
# Cell 8 — the hand-back. One zip: adapter, eval JSON, run config, split.
!mkdir -p release/re-v1 && cp -r runs/re-v1/adapter release/re-v1/ \
  && cp runs/re-v1/run_config.json runs/re-v1/train_summary.json release/re-v1/ \
  && cp runs/eval/before.json runs/eval/after.json release/re-v1/ \
  && cp cloris_boolean_rl/data/splits/re-v1.json release/re-v1/split.json \
  && cp cloris_boolean_rl/scorer/VENDORED_FROM.json release/re-v1/
!cd release && zip -qr cloris-boolean-rl-re-v1.zip re-v1 && ls -la
!cp release/cloris-boolean-rl-re-v1.zip /content/drive/MyDrive/boolean-search-rl/
